<h3>Cấu hình

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import warnings
import duckdb
import gc

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

db_path = Path("database.db")
file_path = Path("input_winmart.xlsx")

def read_excel(file, sheet):
    """
    Đọc các sheet excel từ file input_winmart.xlsx
    Trả về DataFrame
    """
    table_dict = {
        "product": "Sản phẩm",
        "product_price": "Giá gốc",
        "promotion": "Chương trình khuyến mãi",
        "location": "Thông tin điểm giao",
        "customer": "Khách hàng",
        "cost_center": "Cost Center"
    }
    df = pd.read_excel(
        file,
        sheet_name=table_dict[sheet]
    )
    return df

def execute_sql(script):
    """
    Tạo bảng và constraint trong databse
    """
    global conn
    # Đóng connection cũ nếu tồn tại
    try:
        conn.close()
    except:
        pass

    try:
        del conn
    except:
        pass

    gc.collect()
    if db_path.exists():
        db_path.unlink()
    conn = sqlite3.connect(db_path)
    with open(script, "r", encoding="utf-8") as f:
        sql = f.read()

    conn.executescript(sql)
    conn.commit()

    return conn

def import_dtb(df, tablename):
    """
    Làm sạch dữ liệu trong bảng trước khi import
    Chỉ import data từ DataFrame, không phá các contraint của bảng
    Các cột trong DataFrame không cần đúng thứ tự, chỉ cần tên giống nhau
    """
    with sqlite3.connect(db_path) as connection:
        cursor = connection.cursor()
        cursor.execute(f"DELETE FROM {tablename}")
        df.to_sql(
            tablename,
            index=False,
            con=connection,
            if_exists="append"
        )
        connection.commit()

def show_result(db_path):
    """
    Trả về một DataFrame chứa kết qủa import
    """
    with sqlite3.connect(db_path) as conn:
        tables = pd.read_sql("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            AND name NOT LIKE 'sqlite_%'
        """, conn)

        result = []

        for table in tables["name"]:
            query = f"""
                SELECT COUNT(*) AS row_count
                FROM {table}
            """
            count = pd.read_sql(query, conn).iloc[0, 0]
            result.append({
                "table_name": table,
                "row_count": count
            })
    return pd.DataFrame(result)

<h3> Đọc và import

Tạo schema

In [2]:
# Tạo các bảng và constraint
execute_sql("sql_scripts.sql")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'database.db'

Các bảng SẢN PHẨM

In [ ]:
# Đọc sheet sản phẩm
read_product = pd.read_excel(file_path,"Sản phẩm")

# Tách nhóm sản phẩm khỏi bảng sản phẩm và import
import_product_category = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS product_category_key,
        "MSG - DS - NK - ĐV" AS ctg_1,
        "DS - ĐV" AS ctg_2
    FROM (
        SELECT DISTINCT
            "DS - ĐV",
            "MSG - DS - NK - ĐV"
        FROM read_product
    )
""").to_df()

# Import dữ liệu vào bảng product_category
import_dtb(import_product_category, "product_category")

import_product = duckdb.sql("""
    SELECT
        CAST("Mã sản phẩm" as TEXT) AS product_code,
        "Tên sản phẩm" AS product_name,
        product_category_key
    FROM read_product pr
    LEFT JOIN import_product_category prc ON pr."MSG - DS - NK - ĐV" = prc.ctg_1
""").to_df()

# Import dữ liệu vào bảng product
import_dtb(import_product, "product")

Bảng giá và loại hệ thống (Win+/Winmart thường)

In [ ]:
# Đọc sheet Giá gốc
base_price = read_excel(file_path, "product_price")

# Trích xuất dữ liệu loai hệ thống (win+ / winmart) từ bảng giá gốc
import_customer_system = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS system_key,
        "Hệ thống" AS system_name
    FROM (
         SELECT DISTINCT "Hệ thống"
         FROM base_price
         WHERE "Hệ thống" IS NOT NULL
    )
""").to_df()
# Import dữ liệu vào customer_system
import_dtb(import_customer_system, "customer_system")

# Lấy dữ liệu của bảng giá gốc
import_base_price = duckdb.sql("""
    SELECT
        --*,
        CAST("Mã Barcode" AS TEXT) AS barcode,
        CAST("Mã sản phẩm" AS TEXT) AS product_code,
        CAST("Giá gốc" AS numeric) AS base_price,
        system_key
    FROM
        base_price bpr
    LEFT JOIN import_customer_system i ON bpr."Hệ thống" = i."system_name"
""").to_df()
# Import giá gốc vào database
import_dtb(import_base_price, "base_price")

Chương trình khuyến mãi

In [ ]:
# Đọc dữ liệu từ sheet "Chương trình khuyến mãi"
promotion = read_excel(file_path,"promotion")
import_promo_type = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS promo_type_key,
        "Chương trình KM" AS promo_type_name
    FROM (
        SELECT DISTINCT "Chương trình KM"
        FROM promotion
        WHERE "Chương trình KM" IS NOT NULL
    )
""").to_df()
import_dtb(import_promo_type, "promo_type")

# Chi tiết chương trình khuyến mãi
import_promotion_detail = duckdb.sql("""
    WITH pr AS(
        SELECT
            CAST("Tên chương trình" AS TEXT) AS post_name,
            CAST("Hệ thống" AS TEXT) AS system_name,
            CAST(strptime("Ngày áp dụng", '%d/%m/%Y') AS DATE) AS start_date,
            CAST(strptime("Ngày kết thúc", '%d/%m/%Y')  AS DATE) AS end_date,
            CAST("Barcode" AS TEXT) AS barcode,
            CAST("% Giảm giá" AS DOUBLE) AS discount_percentage,
            CAST("Chương trình KM" AS TEXT) AS promo_type_name,
            CAST("Mã hàng tặng" AS TEXT) AS promo_product_name
        FROM promotion
    )
    SELECT
        post_name,
        system_key,
        start_date,
        end_date,
        discount_percentage,
        promo_type_key,
        barcode
    FROM pr
    LEFT JOIN import_customer_system USING (system_name)
    LEFT JOIN import_promo_type USING (promo_type_name)
""").to_df()
import_dtb(import_promotion_detail, "promotion_detail")

Bảng chi nhánh, kho bãi, khách hàng

In [ ]:
# Đọc dữ liệu từ sheet Khách hàng
customer = read_excel(file_path, "customer")

# Lấy dữ liệu mã kho và import
import_warehouse = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS warehouse_key,
        "Mã kho" AS warehouse_code,
    FROM (
        SELECT DISTINCT "Mã kho"
        FROM customer
         )
    customer
""").to_df()
import_dtb(import_warehouse, "warehouse")

# Lấy dữ liệu chi nhánh và import
import_branch = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS branch_key,
        "Chi nhánh" AS branch_name,
    FROM (
        SELECT DISTINCT "Chi nhánh"
        FROM customer
         )
    customer
""").to_df()
import_dtb(import_branch, "branch")

# Join lại dữ liệu 3 bảng dim (warehouse, branch, product_category) vào customer
import_customer = duckdb.sql("""
    WITH customer_r AS (
        SELECT CAST("Mã khách hàng" AS TEXT) AS customer_code,
        "Tên khách hàng" AS customer_name,
        "Chi nhánh" AS branch_name,
        "Nhóm hàng" AS ctg_1,
        "Mã kho" AS warehouse_code
        FROM customer
    )
    SELECT
    -- *,
        customer_code,
        customer_name,
        branch_key,
        warehouse_key,
        product_category_key
    FROM customer_r r
    LEFT JOIN import_warehouse USING (warehouse_code)
    LEFT JOIN import_branch USING (branch_name)
    LEFT JOIN import_product_category USING (ctg_1)
""").to_df()
import_dtb(import_customer, "customer")

Bảng địa điểm giao hàng

In [ ]:
# Đọc dữ liệu từ sheet Thông tin điểm giao
location = read_excel(file_path, "location")

# Đọc dữ liệu và import vào bảng location
import_location = duckdb.sql("""
    WITH location_r AS (
        SELECT
            "Mã điểm giao" AS location_code,
            "Hệ thống" AS system_name,
            "Tên điểm giao" AS location_name,
            "Địa chỉ điểm giao" AS location_address,
            "Mã kho" AS warehouse_code,
            "Nhóm hàng" AS ctg_1,
            "Mã khách hàng" AS customer_code
        FROM location
    )
    SELECT
    location_code,
    location_address,
    system_key,
    customer_code
    FROM location_r
    LEFT JOIN import_customer_system USING (system_name)
    LEFT JOIN import_warehouse USING (warehouse_code)
    LEFT JOIN import_product_category USING(ctg_1)
    --WHERE product_category_key IS NULL
""").to_df()
import_dtb(import_location, "delivery_location")

<h3> Kết quả

In [ ]:
show_result(db_path)